# Cypher Query Examples — `one_rainbow` KG

Quick reference notebook with common Cypher query patterns against the **`one_rainbow`** knowledge graph (Rainbow opportunity reviews).

## Setup
Run the cell below to initialise the environment and connect to the KG.


In [38]:
from genai_tk.utils.config_mngr import global_config
from rich.console import Console
from rich.table import Table

from genai_graph.kg.backend import create_backend_from_config
from genai_graph.kg.embeddings_handler import EmbeddingsHandler

console = Console()

# Connect to the one_rainbow KG output folder
backend = create_backend_from_config("default", "one_rainbow")


def run_query(query: str, title: str = "Results", params: dict | None = None):
    """Execute a Cypher query and display results as a Rich table."""
    try:
        result = backend.execute(query, params or {})
        df = result.get_as_df()

        table = Table(title=f"{title} ({len(df)} rows)")
        for col in df.columns:
            table.add_column(str(col), style="cyan")
        for _, row in df.iterrows():
            table.add_row(*[str(val) for val in row])

        console.print(table)
        return df
    except Exception as e:
        console.print(f"[red]Error: {e}[/red]")
        return None


print("✅ Connected to one_rainbow KG. Ready to run queries!")


✅ Connected to one_rainbow KG. Ready to run queries!


## Basic Queries


In [39]:
# Count nodes by type
query = "MATCH (n) RETURN labels(n)[0] as type, count(n) as count ORDER BY count DESC"
run_query(query, "Node Counts by Type")

 Node Counts by 
 Type (1 rows)  
┏━━━━━━┳━━━━━━━┓
┃ type ┃ count ┃
┡━━━━━━╇━━━━━━━┩
│      │ 33    │
└──────┴───────┘

,type,count
0,,33


In [40]:
# List relationship types
query = "MATCH ()-[r]->() RETURN type(r) as relationship, count(*) as count ORDER BY count DESC"
run_query(query, "Relationship Types")

Error: Catalog exception: function TYPE does not exist.

In [41]:
# Most recent reviewed opportunities
run_query(
    "MATCH (n:ReviewedOpportunity) RETURN n.name, n.start_date ORDER BY n.start_date DESC LIMIT 10",
    "Recent Reviewed Opportunities",
)


    Recent Reviewed Opportunities (1 rows)     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ n.name                       ┃ n.start_date ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ Review:9000559500:2019-04-01 │ 2019-04-01   │
└──────────────────────────────┴──────────────┘

,n.name,n.start_date
0,Review:9000559500:2019-04-01,2019-04-01


## Relationship Queries


In [42]:
# Opportunities with competitors
query = """
MATCH (ro:ReviewedOpportunity)-[:HAS_COMPETITOR]->(c:Competitor)
RETURN ro.name as opportunity, c.name as competitor
ORDER BY ro.name
LIMIT 10
"""
run_query(query, "Opportunities with Competitors")

   Opportunities with Competitors (3 rows)   
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ opportunity                  ┃ competitor ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ Review:9000559500:2019-04-01 │ Capgemini  │
│ Review:9000559500:2019-04-01 │ Thales     │
│ Review:9000559500:2019-04-01 │ Other      │
└──────────────────────────────┴────────────┘

,opportunity,competitor
0,Review:9000559500:2019-04-01,Capgemini
1,Review:9000559500:2019-04-01,Thales
2,Review:9000559500:2019-04-01,Other


In [43]:
# Reviews → Opportunity → Customer
run_query(
    """
    MATCH (ro:ReviewedOpportunity)-[:REVIEWS]->(opp:Opportunity)-[:HAS_CUSTOMER]->(cust:Customer)
    RETURN ro.name AS review, opp.name AS opportunity, cust.name AS customer
    ORDER BY ro.name
    LIMIT 10
    """,
    "Reviews → Opportunity → Customer",
)


                      Reviews → Opportunity → Customer (1 rows)                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ review                       ┃ opportunity                             ┃ customer ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ Review:9000559500:2019-04-01 │ CNES TMA VENUS VIP, PEPS, THEIA MUSCATE │ CNES     │
└──────────────────────────────┴─────────────────────────────────────────┴──────────┘

,review,opportunity,customer
0,Review:9000559500:2019-04-01,"CNES TMA VENUS VIP, PEPS, THEIA MUSCATE",CNES


In [44]:
# Count competitors per opportunity
query = """
MATCH (ro:ReviewedOpportunity)-[:HAS_COMPETITOR]->(c:Competitor)
RETURN ro.name as opportunity, count(c) as competitor_count
ORDER BY competitor_count DESC
LIMIT 10
"""
run_query(query, "Competitors per Opportunity")

       Competitors per Opportunity (1 rows)        
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ opportunity                  ┃ competitor_count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Review:9000559500:2019-04-01 │ 3                │
└──────────────────────────────┴──────────────────┘

,opportunity,competitor_count
0,Review:9000559500:2019-04-01,3


## Statistical Queries


In [45]:
# Financial statistics
query = """
MATCH (n:ReviewedOpportunity)
RETURN 
  count(n) as total_count,
  avg(n.financials.tcv) as avg_tcv,
  max(n.financials.tcv) as max_tcv,
  min(n.financials.tcv) as min_tcv
"""
run_query(query, "Financial Statistics")

           Financial Statistics (1 rows)           
┏━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ total_count ┃ avg_tcv   ┃ max_tcv   ┃ min_tcv   ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━┩
│ 1.0         │ 2380000.0 │ 2380000.0 │ 2380000.0 │
└─────────────┴───────────┴───────────┴───────────┘

,total_count,avg_tcv,max_tcv,min_tcv
0,1,2380000.0,2380000.0,2380000.0


In [46]:
# Risk category distribution
run_query(
    """
    MATCH (r:RiskAnalysis)
    RETURN r.risk_category AS category, count(r) AS count
    ORDER BY count DESC
    """,
    "Risk Categories",
)


   Risk Categories (5 rows)    
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ category            ┃ count ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ DeliveryCostOverrun │ 1     │
│ ScopeManagementRisk │ 1     │
│ SkillRetentionRisk  │ 1     │
│ SLAComplianceRisk   │ 1     │
│ nan                 │ 1     │
└─────────────────────┴───────┘

,category,count
0,DeliveryCostOverrun,1
1,ScopeManagementRisk,1
2,SkillRetentionRisk,1
3,SLAComplianceRisk,1
4,NaN,1


## Embedding / Vector Search

The rainbow schema stores vectorised embeddings for several node fields.
Embedding columns follow the `{field}_embedding` naming convention; vector
indexes are named `{field}_index`.

| Node | Indexed field | Index name |
|---|---|---|
| `Opportunity` | `name` | `name_index` |
| `Customer` | `name` | `name_index` |
| `RiskAnalysis` | `risk_description` | `risk_description_index` |
| `TechnicalApproach` | `architecture` | `architecture_index` |
| `TechnicalApproach` | `technical_stack` | `technical_stack_index` |

> The same embeddings model configured in `kg_build.embeddings.default` is used
> at build time **and** here, so query vectors are in the same space.


In [47]:
# Initialise the embeddings handler with the model used at KG build time
cfg = global_config()
embeddings_id = cfg.get_str("kg_build.embeddings.default")
print(f"Embeddings model: {embeddings_id}")

emb_handler = EmbeddingsHandler(embeddings_id=embeddings_id)
print("EmbeddingsHandler ready.")


2026-02-26 19:37:01.638 | DEBUG    | genai_tk.core.embeddings_factory:get_embeddings:571 - get embeddings: 'qwen3_06b@deepinfra' -cache: True
2026-02-26 19:37:01.688 | DEBUG    | genai_graph.kg.embeddings_handler:__init__:49 - EmbeddingsHandler initialized with model: qwen3_06b@deepinfra


Embeddings model: qwen3_06b@deepinfra
EmbeddingsHandler ready.


In [48]:
# Vector search: find Opportunities whose name is closest to a free-text query
query_text = "cloud infrastructure modernisation"
query_vector = emb_handler.compute_embeddings(query_text)

result = backend.query_vector_index(
    table_name="Opportunity",
    index_name="name_index",
    query_vector=query_vector,
    k=5,
)
df = result.get_as_df()
console.print(f"\n[bold]Top-5 Opportunities similar to:[/bold] '{query_text}'")
console.print(df.to_string(index=False))


2026-02-26 19:37:02.517 | DEBUG    | genai_graph.kg.embeddings_handler:compute_embeddings:73 - Computed embedding for text (34 chars) -> 1024 dims
2026-02-26 19:37:02.541 | DEBUG    | genai_graph.kg.backend:query_vector_index:462 - Vector index query returned results


Top-5 Opportunities similar to: 'cloud infrastructure modernisation'

Empty DataFrame
Columns: 
Index: []

In [49]:
# Vector search on RiskAnalysis descriptions
query_text = "supply chain delivery delay"
query_vector = emb_handler.compute_embeddings(query_text)

result = backend.query_vector_index(
    table_name="RiskAnalysis",
    index_name="risk_description_index",
    query_vector=query_vector,
    k=5,
)
df = result.get_as_df()
console.print(f"\n[bold]Top-5 Risks similar to:[/bold] '{query_text}'")
console.print(df.to_string(index=False))


2026-02-26 19:37:03.289 | DEBUG    | genai_graph.kg.embeddings_handler:compute_embeddings:73 - Computed embedding for text (27 chars) -> 1024 dims
2026-02-26 19:37:03.295 | DEBUG    | genai_graph.kg.backend:query_vector_index:462 - Vector index query returned results


Top-5 Risks similar to: 'supply chain delivery delay'

Empty DataFrame
Columns: 
Index: []

In [50]:
# Combined: vector search on RiskAnalysis, then traverse back to ReviewedOpportunity
query_text = "security compliance regulatory"
query_vector = emb_handler.compute_embeddings(query_text)

risk_result = backend.query_vector_index(
    table_name="RiskAnalysis",
    index_name="risk_description_index",
    query_vector=query_vector,
    k=10,
)
risk_df = risk_result.get_as_df()

# Extract node names from the returned node maps
risk_names = [row["node"].get("name", "") for _, row in risk_df.iterrows() if isinstance(row["node"], dict)]

if risk_names:
    run_query(
        """
        MATCH (ro:ReviewedOpportunity)-[:HAS_RISK]->(r:RiskAnalysis)
        WHERE r.name IN $names
        RETURN ro.name AS opportunity, r.name AS risk, r.risk_description AS description
        ORDER BY ro.name
        LIMIT 10
        """,
        f"Opportunities with risks related to '{query_text}'",
        params={"names": risk_names},
    )
else:
    console.print("[yellow]No matching risk nodes found.[/yellow]")


2026-02-26 19:37:04.082 | DEBUG    | genai_graph.kg.embeddings_handler:compute_embeddings:73 - Computed embedding for text (30 chars) -> 1024 dims
2026-02-26 19:37:04.089 | DEBUG    | genai_graph.kg.backend:query_vector_index:462 - Vector index query returned results


No matching risk nodes found.

## Quick Reference

### Common Cypher Patterns

**Count nodes:**
```cypher
MATCH (n:Label) RETURN count(n)
```

**Find with property:**
```cypher
MATCH (n:Label) WHERE n.property = 'value' RETURN n
```

**Traverse relationships:**
```cypher
MATCH (a)-[:REL_TYPE]->(b) RETURN a, b
```

**Access MAP / STRUCT field:**
```cypher
MATCH (n:ReviewedOpportunity) RETURN n.financials.tcv
```

### Vector Search Pattern

```python
# 1. Compute query vector with the same model used at build time
query_vector = emb_handler.compute_embeddings("your query text")

# 2. Query a vector index
result = backend.query_vector_index(
    table_name="NodeLabel",
    index_name="field_index",   # {field}_index convention
    query_vector=query_vector,
    k=10,
)
df = result.get_as_df()   # columns: node (dict), distance (float)
```

### Rainbow KG Schema

```
ReviewedOpportunity
  -[:REVIEWS]->          Opportunity       (name_embedding ✦)
  -[:HAS_RISK]->         RiskAnalysis      (risk_description_embedding ✦)
  -[:HAS_COMPETITOR]->   Competitor
  -[:HAS_PARTNER]->      Partner
  -[:HAS_TEAM_MEMBER]->  Person
  -[:DELIVERED_IN]->     Geo
Opportunity
  -[:HAS_CUSTOMER]->     Customer          (name_embedding ✦)
  -[:HAS_CONTACT]->      Person
TechnicalApproach        (architecture_embedding ✦, technical_stack_embedding ✦)

✦ = vector index available
```
